# Text → Vision Attention Maps (PaliGemma) — Colab demo

This notebook:
1. Clones the repo from GitHub.
2. Loads **PaliGemma** straight from the HuggingFace Hub.
3. Takes **any image URL** from the internet + **any question** you type.
4. Extracts the decoder self-attention at **layers 0 and 1** and slices it so that
   **rows = text tokens** and **columns = visual tokens**.

> **Runtime:** use a GPU runtime (`Runtime → Change runtime type → GPU`). PaliGemma is a
> **gated** model — accept its license at
> https://huggingface.co/google/paligemma-3b-pt-224 with the same account you log in with.

## 1. Install dependencies

In [ ]:
!pip -q install -U transformers huggingface_hub safetensors pillow

## 2. Clone the repo

In [ ]:
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git
%cd text_vision_attention_map

## 3. Authenticate with HuggingFace
PaliGemma is gated. Accept the license once (link above), then run this cell and paste a token
(create one at https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Pick an image (any URL) and a question

In [ ]:
import requests
from PIL import Image

# ---- change these two lines to anything you like ----
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"  # any image URL
question  = "how many cats are in the image?"                          # any question
# -----------------------------------------------------

image_path = "/content/input_image.jpg"
with open(image_path, "wb") as f:
    f.write(requests.get(image_url, timeout=30).content)

img = Image.open(image_path).convert("RGB")
print("Question:", question)
img

## 5. Load PaliGemma (downloads from the Hub on first run)

In [ ]:
from get_attention_map import load_paligemma, get_attention_maps

model, processor, device = load_paligemma("google/paligemma-3b-pt-224")
print("Loaded on:", device)

## 6. (Optional) Run normal inference to see the model's answer

In [ ]:
from inference import generate

answer = generate(model, processor, device, question, image_path, max_tokens=20)
print("Model answer:", answer)

## 7. Extract attention at decoder layers 0 and 1
`P[layer]` has shape `[L_t, L_v]` — **rows = text tokens, columns = visual tokens.**

In [ ]:
out = get_attention_maps(model, processor, device, question, image_path, layers=[0, 1])

print("L_t (text rows)  :", len(out["text_tokens"]))
print("L_v (vision cols):", out["vision_positions"].numel())
print("row (text) labels:", out["text_tokens"])
for L in [0, 1]:
    print(f"layer {L}: P shape = {tuple(out['maps'][L].shape)}  (rows=text, cols=vision)")

## 8. Visualize the maps (rows = text tokens, columns = visual tokens)

In [ ]:
import matplotlib.pyplot as plt

for L in [0, 1]:
    P = out["maps"][L]                 # [L_t, L_v], averaged over heads
    tokens = out["text_tokens"]
    fig, ax = plt.subplots(figsize=(12, max(3, len(tokens) * 0.4)))
    im = ax.imshow(P.numpy(), aspect="auto", cmap="viridis")
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_xlabel(f"visual tokens (columns)  —  L_v = {P.shape[1]}")
    ax.set_ylabel("text tokens (rows)")
    ax.set_title(f"Decoder layer {L}:  text → vision attention  P")
    fig.colorbar(im, ax=ax, fraction=0.02)
    plt.show()

### (Optional) See where one word looks on the image
The 256 visual columns form a 16×16 grid. Reshape one text row and overlay it on the image.

In [ ]:
import numpy as np
import math

layer = 0
word_row = len(out["text_tokens"]) - 1   # last prompt token; change to any row index

P = out["maps"][layer]
L_v = P.shape[1]
g = int(math.sqrt(L_v))                    # 16 for PaliGemma-224
heat = P[word_row].numpy().reshape(g, g)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img); ax[0].set_title("image"); ax[0].axis("off")
ax[1].imshow(img)
ax[1].imshow(np.array(Image.fromarray((heat / heat.max() * 255).astype('uint8')).resize(img.size)),
             cmap="jet", alpha=0.5)
ax[1].set_title(f"layer {layer}, token '{out['text_tokens'][word_row]}'"); ax[1].axis("off")
plt.show()